<a href="https://colab.research.google.com/github/mk654/SML_PG60/blob/main/COMP90051_ProjectGroup60_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [ ]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np

# Amelia -> load data from github repo
!git clone https://github.com/mk654/SML_PG60
repo_dir = Path("/content/SML_PG60")

flu_mat = repo_dir / "data" / "matraw" / "influenza_outbreak_dataset.mat" # access influenza dataset from gitrepo
fludata = loadmat(flu_mat)



# url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat" # old access -> directory = main
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat" # old access -> directory = main/data


Cloning into 'SML_PG60'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 4.33 MiB | 9.13 MiB/s, done.
Resolving deltas: 100% (14/14), done.


### 0.a) inspecting data

*Amelia*


#### *Results*
| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

in sum:
- 48 training feature matrices, one per location
- 48 testing feature matrices, one per location
- 48 training label vectors
- 48 testing label vectors
- 48 location names/IDs
- 545 keyword feature names



*Code*


```
rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)
```

In [ ]:
# delete this cell before submission!!

rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)

           name outer_dtype outer_shape inner_type  inner_shape
0      flu_X_tr      object     (1, 48)  csc_array  (1095, 545)
1      flu_X_te      object     (1, 48)  csc_array   (485, 545)
2      flu_Y_tr      object     (1, 48)    ndarray    (1095, 1)
3      flu_Y_te      object     (1, 48)    ndarray     (485, 1)
4      flu_locs      object     (1, 48)    ndarray         (1,)
5  flu_keywords      object    (1, 525)    ndarray         (1,)


### 0.b) convert .mat files to .csv
Converts influenza_outbreak_dataset.mat to csv
- github directory = ```SML_PG60/data/processed/flu_csv```
- conversion will produce 48 separate folders aligning with the 48 folds. Each folder will contain:
  1. X_train.csv
  2. X_test.csv
  3. y_train.csv
  4. y_test.csv
  * *note that influenza_outbreak_dataset.mat contains 48 folds (test/train splits). Each fold contains its own X & y train and X & y test. Hence, the data is split and converted as cleanly as possible to avoid errors that may come with combining folds*
- Additionally, the conversion will also produce:
  1. keywords.csv
  2. locs.csv



----
*Note that these csv files are only processed in the sense that they have been converted from .mat to .csv; no further processing has yet taken place.*

In [ ]:
# Amelia -> convert influenza_outbreak_dataset.mat to .csv file
flu_out = repo_dir / "data" / "processed" / "flu_csv"
flu_out.mkdir(parents=True, exist_ok=True)

X_tr = fludata["flu_X_tr"]
X_te = fludata["flu_X_te"]
y_tr = fludata["flu_Y_tr"]
y_te = fludata["flu_Y_te"]

n_folds = X_tr.shape[1]

for i in range(n_folds):
    fold_dir = flu_out / f"fold_{i:02d}"
    fold_dir.mkdir(exist_ok=True)

    # extract cells
    Xtr = X_tr[0, i].toarray() # some data stored as sparse matrix, convert to dense
    Xte = X_te[0, i].toarray()
    ytr = y_tr[0, i].ravel()
    yte = y_te[0, i].ravel()

    # convert to dataframe
    pd.DataFrame(Xtr).to_csv(fold_dir / "X_train.csv", index=False)
    pd.DataFrame(Xte).to_csv(fold_dir / "X_test.csv", index=False)
    pd.DataFrame(ytr).to_csv(fold_dir / "y_train.csv", index=False)
    pd.DataFrame(yte).to_csv(fold_dir / "y_test.csv", index=False)

    print(f"Saved fold {i}")

# keywords
keywords = fludata["keywords"]

# flatten and extract strings
keywords_list = [str(k[0]) for k in keywords.ravel()]
pd.DataFrame(keywords_list, columns=["keyword"]) \
  .to_csv(flu_out / "keywords.csv", index=False)

# locs
locs = fludata["locs"]

locs_list = [str(l[0]) for l in locs.ravel()]
pd.DataFrame(locs_list, columns=["location"]) \
  .to_csv(flu_out / "locs.csv", index=False)